# ICOS metadata and data access

Note: In order to use the ICOS Python library (<code>icoscp_core</code>) to download data, you first need to be authenticated. Follow the instructions at [https://icos-carbon-portal.github.io/pylib/icoscp/authentication/](https://icos-carbon-portal.github.io/pylib/icoscp/authentication/).

In [1]:
! pip install icoscp_core

! pip list

Package                   Version
------------------------- ------------
aiohappyeyeballs          2.6.1
aiohttp                   3.12.15
aiosignal                 1.4.0
annotated-types           0.7.0
anyio                     4.13.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
argopy                    1.4.0
asttokens                 3.0.1
async-timeout             5.0.1
attrs                     26.1.0
Brotli                    1.2.0
cached-property           1.5.2
certifi                   2026.4.22
cffi                      2.0.0
cftime                    1.6.5
chardet                   7.4.3
charset-normalizer        3.4.7
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
dacite                    1.8.1
debugpy                   1.8.20
decorator                 5.2.1
envrihub                  0.1.3
erddapy                   3.1.1
exceptiongroup            1.3.1
executing                 2.2.1
fonttools          

In [2]:
from datetime import datetime
import requests
import pandas as pd
from icoscp_core.icos import meta, data

## User-defined filters

In [3]:
exv = "EXV013"
dt_min = datetime(2020, 1, 1)
dt_max = datetime(2025, 1, 1)
sampling_height_min = 0
sampling_height_max = 100
latitude_min = 45
latitude_max = 65
longitude_min = 10
longitude_max = 20

dt_format = "%Y-%m-%dT%H:%M:%SZ"

#### Format filters so they can be inserted into the SPARQL query

In [4]:
spatial_filters = {
    "?lon > ": longitude_min, "?lon < ": longitude_max, "?lat > ": latitude_min, "?lat < ": latitude_max,
    "?height > ": sampling_height_min, "?height < ": sampling_height_max
}
filters = []
if dt_min is not None:
    filters.append(f"?end > '{datetime.strftime(dt_min, dt_format)}'^^xsd:dateTime")
if dt_max is not None:
    filters.append(f"?start < '{datetime.strftime(dt_max, dt_format)}'^^xsd:dateTime")
for filter_str, filter_val in spatial_filters.items():
    if filter_val is not None:
        filters.append(f"{filter_str}{filter_val}")

## Fetch ICOS data objects matching the selected EXV and the filters

#### Get URIs of relevant variables from NVS P07 vocabulary
Relevant variables are those which have the same I-ADOPT annotations as the selected EXV. They are found by querying the NVS SPARQL endpoint.

In [5]:
query = """
PREFIX skos:   <http://www.w3.org/2004/02/skos/core#>
PREFIX iop:    <https://w3id.org/iadopt/ont#>
SELECT ?p07 WHERE {
  VALUES ?exv { <http://vocab.nerc.ac.uk/collection/EXV/current/%s/> }
  ?exv iop:hasApplicableProperty ?prop .
  ?exv iop:hasApplicableObjectOfInterest ?ooi .
  ?exv iop:hasApplicableMatrix ?matrix .
  <http://vocab.nerc.ac.uk/collection/P07/current/> skos:member ?p07 .
  ?p07 iop:hasProperty ?prop .
  ?p07 iop:hasObjectOfInterest ?ooi .
  ?p07 iop:hasMatrix ?matrix .
}
""" % exv
response = requests.get("http://vocab.nerc.ac.uk/sparql/sparql", params={"query": query}, headers={"Accept": "application/sparql-results+json"})
response.raise_for_status()
results = response.json()
p07_vars = [binding["p07"]["value"] for binding in results["results"]["bindings"]]
print(p07_vars)
if len(p07_vars) == 0:
    raise Exception(f"No P07 variable matching the selected EXV ({exv}) was found")

['http://vocab.nerc.ac.uk/collection/P07/current/CFV7N47/', 'http://vocab.nerc.ac.uk/collection/P07/current/CFV7N55/', 'http://vocab.nerc.ac.uk/collection/P07/current/CFV8N44/', 'http://vocab.nerc.ac.uk/collection/P07/current/GAXBFKR9/', 'http://vocab.nerc.ac.uk/collection/P07/current/760D4K7J/', 'http://vocab.nerc.ac.uk/collection/P07/current/M8TGRULE/']


#### Build a SPARQL query for the ICOS SPARQL endpoint
The query takes as inputs:
- a list of URIs to P07 variables (obtained from the previous step),
- a list of temporal and spatial filters (defined earlier)

The query will select ICOS datasets which contain data about variables that are mapped (in the ICOS ontology) to any of the listed P07 variables using <code>skos:exactMatch</code>.

Note: only most recent (i.e. not deprecated) ICOS level 2 data objects are returned.

In [6]:
p07_vars_str = "<" + "> <".join(p07_vars) + ">"
filters_str = " && ".join(filters)
query = """
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX cpmeta: <http://meta.icos-cp.eu/ontologies/cpmeta/>
#PREFIX geo: <http://www.opengis.net/ont/geosparql#>

SELECT ?dobj
WHERE {
	VALUES ?externalVar { %s }
    ?valueType skos:exactMatch ?externalVar .
    ?spec cpmeta:containsDataset/cpmeta:hasColumn/cpmeta:hasValueType ?valueType .
    ?spec cpmeta:hasDataLevel ?level .
    ?dobj cpmeta:hasObjectSpec ?spec .
    ?dobj cpmeta:wasAcquiredBy ?acq .
	?dobj cpmeta:hasStartTime | (cpmeta:wasAcquiredBy/prov:startedAtTime) ?start .
	?dobj cpmeta:hasEndTime | (cpmeta:wasAcquiredBy/prov:endedAtTime) ?end .
    ?acq prov:wasAssociatedWith ?station .
    ?acq cpmeta:hasSamplingHeight ?height .
    ?station cpmeta:hasLatitude ?lat .
    ?station cpmeta:hasLongitude ?lon .
    FILTER ( %s )
    FILTER NOT EXISTS {[] cpmeta:isNextVersionOf ?dobj}
    FILTER (?level = 2)
}
""" % (p07_vars_str, filters_str)
print(query)


PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX cpmeta: <http://meta.icos-cp.eu/ontologies/cpmeta/>
#PREFIX geo: <http://www.opengis.net/ont/geosparql#>

SELECT ?dobj
WHERE {
	VALUES ?externalVar { <http://vocab.nerc.ac.uk/collection/P07/current/CFV7N47/> <http://vocab.nerc.ac.uk/collection/P07/current/CFV7N55/> <http://vocab.nerc.ac.uk/collection/P07/current/CFV8N44/> <http://vocab.nerc.ac.uk/collection/P07/current/GAXBFKR9/> <http://vocab.nerc.ac.uk/collection/P07/current/760D4K7J/> <http://vocab.nerc.ac.uk/collection/P07/current/M8TGRULE/> }
    ?valueType skos:exactMatch ?externalVar .
    ?spec cpmeta:containsDataset/cpmeta:hasColumn/cpmeta:hasValueType ?valueType .
    ?spec cpmeta:hasDataLevel ?level .
    ?dobj cpmeta:hasObjectSpec ?spec .
    ?dobj cpmeta:wasAcquiredBy ?acq .
	?dobj cpmeta:hasStartTime | (cpmeta:wasAcquiredBy/prov:st

#### POST the SPARQL query to the ICOS SPARQL endpoint

This returns a list of URLs to metadata pages of datasets which contain data related to at least one of the P07 variables and which are matching the spatial and temporal filters.

In [7]:
resp = requests.post("https://meta.icos-cp.eu/sparql", query)
bindings = resp.json()["results"]["bindings"]
dobjs = [binding["dobj"]["value"] for binding in bindings]
print(len(dobjs), dobjs[:100], sep="\n")

246
['https://meta.icos-cp.eu/objects/6pLNj6DdaSqhlLKuQJS0qxsa', 'https://meta.icos-cp.eu/objects/ZU4dJG4OYWv052zn4YX35K9K', 'https://meta.icos-cp.eu/objects/FunTAtWVhA81A6gXgN-O0lnq', 'https://meta.icos-cp.eu/objects/71m-_WbKKMhxgFqtdeGX2ufr', 'https://meta.icos-cp.eu/objects/__SArJfAjTsKTXUAj12OBmws', 'https://meta.icos-cp.eu/objects/N_r3zUlWqY7Nmlmi86Ty8-CR', 'https://meta.icos-cp.eu/objects/ZTiVWtWXYkBzqR5A5KEMooG-', 'https://meta.icos-cp.eu/objects/rkNaKDVvCZfedPlr7qWSL6YE', 'https://meta.icos-cp.eu/objects/6V0tZfVg3lMjd3aedJZrAfP2', 'https://meta.icos-cp.eu/objects/ttVCWSrHPKUnPu8lKJifYmV7', 'https://meta.icos-cp.eu/objects/ySm70SAyVljBO8jK3TlwwaB-', 'https://meta.icos-cp.eu/objects/5A7_kSn8u2_ltIXPERMfZwpl', 'https://meta.icos-cp.eu/objects/wkMK0TpDAFjnheU2DAA8rKA3', 'https://meta.icos-cp.eu/objects/7arVt77z_somqD4zlhkWsmC9', 'https://meta.icos-cp.eu/objects/YO_XMwtRDUmTZ9QXv-EFEIGX', 'https://meta.icos-cp.eu/objects/E_dl2d-s8Sy72SmZXdNAsXsE', 'https://meta.icos-cp.eu/objects/Ip

#### Fetch the data corresponding to the first dataset in the list of URLs

In [8]:
dataset = data.get_columns_as_arrays(meta.get_dobj_meta(dobjs[0]))

Exception: Authentication config file not found at /home/jovyan/.icoscp/cpauthToken_auth_conf.json. Please initialize authentication according to https://github.com/ICOS-Carbon-Portal/data/tree/master/src/main/python/icoscp_core#authentication 
(Remember to use Repository-specific imports in 'from icoscp_core.<repo> import auth')

#### Convert the results to a pandas DataFrame for visualisation purpose

In [ ]:
dataset_df = pd.DataFrame(dataset)
print(dataset_df)

#### Fetch data from all datasets
Additional information on how to use the resulting iterator is provided at [https://icos-carbon-portal.github.io/pylib/icoscp/getting_started/#batch-data-access](https://icos-carbon-portal.github.io/pylib/icoscp/getting_started/#batch-data-access).

In [ ]:
dobjs_data = data.batch_get_columns_as_arrays(dobjs)